In [ ]:
import xarray as xr
import dask.dataframe as dd
import os
from data_sparsity.generate_data import GenerateData
ncpath = "./tutorial1/netCDF/ds.nc"
pqpath = "./tutorial1/parquet/ddf"
pqpathtmp = "./tutorial1/parquet/tmp/"

### List of cases - Single variable

2D because it's easier to understand and visualize

* Purely gridded data, maximum density (minimum sparsity)
* Purely irregular data
* Maximum sparsity on a grid
* Somehow sparse data on a grid

### 3x3 grid, 9 points

In [ ]:
gen = GenerateData(
    num_obs=9,
    num_dims=2,
    ratio_dims=1,
    sparsity=1,  # Safely above minimum for this configuration
    seed=101
)

In [ ]:
ds,df = gen.generate(
    netcdf_filepath = ncpath,
    parquet_filepath = pqpath,
    parquet_tmp = pqpathtmp,
)

In [ ]:
ds = xr.open_dataset(
    ncpath
)
ds

In [ ]:
ds.count()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x0 = ds["x0"].values
x1 = ds["x1"].values
support_x1, support_x0 = np.meshgrid(x1, x0)
present_mask = np.isfinite(ds["record"].values)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(
    support_x1.ravel(),
    support_x0.ravel(),
    s=160,
    facecolors="none",
    edgecolors="dimgray",
    linewidths=1.2,
    zorder=2,
)

for xv in x1:
    ax.axvline(xv, color="dimgray", linestyle=":", linewidth=1, zorder=0)
for yv in x0:
    ax.axhline(yv, color="dimgray", linestyle=":", linewidth=1, zorder=0)

ax.scatter(
    support_x1.ravel()[present_mask.ravel()],
    support_x0.ravel()[present_mask.ravel()],
    marker="x",
    c="green",
    s=90,
    linewidths=2,
    zorder=3,
)

ax.set_xticks(x1)
ax.set_yticks(x0)
ax.set_xticklabels([f"{value:.3f}" for value in x1], color="dimgray")
ax.set_yticklabels([f"{value:.3f}" for value in x0], color="dimgray")
ax.set_xlabel("x1", color="dimgray")
ax.set_ylabel("x0", color="dimgray")
ax.tick_params(axis="both", colors="dimgray")
ax.set_aspect("equal", adjustable="box")
for spine in ax.spines.values():
    spine.set_color("dimgray")
ax.grid(False)
plt.tight_layout()
plt.show()


In [ ]:
ddf = dd.read_parquet(os.path.dirname(pqpath))

In [ ]:
ddf.head()

In [ ]:
ddf.count().compute()

In [ ]:
ddf.nunique().compute()

In [ ]:
ddf['x0'].compute()

In [ ]:
ddf['x1'].compute()

In [ ]:
ddf['record'].compute()

In [ ]:
len(ddf['record'].drop_duplicates().compute())